# 基于标准化设备 Log 的 AI 良率异常洞察 Demo

本 Notebook 展示一个从设备 Log 到工程洞察的轻量级 AI 良率分析流程。

核心流程包括：

1. 读取原始设备 Log；
2. 将原始 Log 标准化为统一分析表；
3. 基于标准化数据进行特征选择与清洗；
4. 构造工艺状态特征；
5. 自动识别异常工况；
6. 对比正常段与异常段，定位关键异常关联因子；
7. 使用 LLM 生成工程师可读的异常分析报告。

重要原则：

- 原始数据只用于生成标准化数据；
- 后续所有分析均基于 `data/processed_equipment_log_standardized.xlsx`；
- Python / 统计 / 机器学习负责计算；
- LLM 只负责解释、总结和生成文本报告。

In [26]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CURRENT_DIR = Path.cwd()
ROOT_CANDIDATES = [CURRENT_DIR, *CURRENT_DIR.parents]
PROJECT_ROOT = next(
    (
        p for p in ROOT_CANDIDATES
        if (p / "data").exists() and (p / "src").exists()
    ),
    CURRENT_DIR,
)

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

RAW_DATA_CANDIDATES = [
    DATA_DIR / "equipment_log.xlsx",
    PROJECT_ROOT / "notebooks" / "equipment_log.xlsx",
    CURRENT_DIR / "data" / "equipment_log.xlsx",
    CURRENT_DIR.parent / "data" / "equipment_log.xlsx",
]
RAW_DATA_PATH = next((p for p in RAW_DATA_CANDIDATES if p.exists()), RAW_DATA_CANDIDATES[0])
PROCESSED_DATA_PATH = DATA_DIR / "processed_equipment_log_standardized.xlsx"

print("当前阶段：1. 环境准备与路径配置")
print("当前工作目录：", CURRENT_DIR)
print("项目根目录：", PROJECT_ROOT)
print("数据目录：", DATA_DIR.resolve())
print("原始数据候选路径：")
for c in RAW_DATA_CANDIDATES:
    print(" -", c)
print("当前选用原始数据路径：", RAW_DATA_PATH)
print("标准化数据文件：", PROCESSED_DATA_PATH)

当前阶段：1. 环境准备与路径配置
当前工作目录： /Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/notebooks
项目根目录： /Users/jiaoguodong/Desktop/myApp/Industry_AIDemo
数据目录： /Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data
原始数据候选路径：
 - /Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data/equipment_log.xlsx
 - /Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/notebooks/equipment_log.xlsx
 - /Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/notebooks/data/equipment_log.xlsx
 - /Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data/equipment_log.xlsx
当前选用原始数据路径： /Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data/equipment_log.xlsx
标准化数据文件： /Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data/processed_equipment_log_standardized.xlsx


In [27]:
print("当前阶段：2. 原始设备 Log 数据读取与结构检查")
print(f"输入数据：{RAW_DATA_PATH}")

if not RAW_DATA_PATH.exists():
    candidate_msg = "\n".join([f" - {p}" for p in RAW_DATA_CANDIDATES])
    raise FileNotFoundError(
        f"未找到原始设备 Log 文件。当前尝试路径：\n{candidate_msg}\n"
        "请确认原始数据文件已放入 data/ 目录（项目根目录或当前工作目录上级目录）。"
    )

df_raw = pd.read_excel(RAW_DATA_PATH)

print(f"原始数据维度：{df_raw.shape[0]} 行 × {df_raw.shape[1]} 列")
print("原始字段示例：", df_raw.columns[:12].tolist())

duplicate_cols = pd.Series(df_raw.columns).value_counts()
duplicate_cols = duplicate_cols[duplicate_cols > 1]
print(f"重复字段数量：{duplicate_cols.shape[0]}")
if not duplicate_cols.empty:
    display(duplicate_cols.to_frame("count").head(20))

l1_cols = [c for c in df_raw.columns if str(c).startswith("L1")]
l2_cols = [c for c in df_raw.columns if str(c).startswith("L2")]
print(f"L1 字段数量：{len(l1_cols)}")
print(f"L2 字段数量：{len(l2_cols)}")
if l1_cols and l2_cols:
    l1_suffix = {c[2:] for c in l1_cols}
    l2_suffix = {c[2:] for c in l2_cols}
    print(f"L2 相对 L1 额外字段数量：{len(l2_suffix - l1_suffix)}")
    print(f"L1 相对 L2 额外字段数量：{len(l1_suffix - l2_suffix)}")

display(df_raw.head())

当前阶段：2. 原始设备 Log 数据读取与结构检查
输入数据：/Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data/equipment_log.xlsx
原始数据维度：1496 行 × 173 列
原始字段示例： ['Time', 'L1 Mode', 'L1 Product Name', 'L1 Lot ID', 'L1 Tool ID', 'L1 Electrode ID', 'L1 Standard Speed', 'L1 Target Speed', 'L1 Running Speed', 'L1 Pump(1) Speed(L)', 'L1 Pump(1) Speed(H)', 'L1 Pump(2) Speed(L)']
重复字段数量：0
L1 字段数量：66
L2 字段数量：68
L2 相对 L1 额外字段数量：5
L1 相对 L2 额外字段数量：3


,Time,L1 Mode,L1 Product Name,L1 Lot ID,L1 Tool ID,L1 Electrode ID,L1 Standard Speed,L1 Target Speed,L1 Running Speed,L1 Pump(1) Speed(L),...,RDI(1) Cond,Cu Strike Level,Ag Strike Level,Ag Spot Level,Ag Spot pH,Ag Deplating(1&2) pH,Ag Deplating(3) pH,RO(1) Pressure,DI(1) Pressure,DI(1) Pressure.1
0,00:00:52,L1:Motion Only,1,NaN,NaN,NaN,0,2.2,0.0,30,...,1.002515,0,0,0,7.066441,8.038797,4.920165,6.012122,1.141699,1.117419
1,00:01:53,L1:Motion Only,1,NaN,NaN,NaN,0,2.2,0.0,30,...,0.997828,0,0,0,7.066441,8.039463,4.920824,5.955643,1.089403,1.043748
2,00:02:54,L1:Motion Only,1,NaN,NaN,NaN,0,2.2,0.0,30,...,0.997828,0,0,0,7.066441,8.038797,4.920824,6.009680,1.140661,1.078405
3,00:03:55,L1:Motion Only,1,NaN,NaN,NaN,0,2.2,0.1,30,...,1.013453,0,0,0,7.066441,8.039463,4.920165,6.019934,1.158923,1.149585
4,00:04:56,L1:Motion Only,1,NaN,NaN,NaN,0,2.2,0.2,30,...,1.013453,0,0,0,7.066441,8.029465,4.920824,6.022701,1.140661,1.162036


In [28]:
print("当前阶段：3. 数据标准化处理层")
print(f"输入数据：{RAW_DATA_PATH}")
print(f"输出数据：{PROCESSED_DATA_PATH}")

def make_unique_columns(columns):
    seen = {}
    out = []
    for col in columns:
        col = str(col).strip()
        if col not in seen:
            seen[col] = 0
            out.append(col)
        else:
            seen[col] += 1
            out.append(f"{col}__dup{seen[col]}")
    return out

def normalize_name(name):
    s = str(name).strip()
    s = s.replace("（", "(").replace("）", ")")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^0-9a-zA-Z_\u4e00-\u9fa5]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def infer_time_columns(df):
    keywords = ["time", "date", "timestamp", "datetime", "record", "log"]
    return [c for c in df.columns if any(k in str(c).lower() for k in keywords)]

def coerce_numeric_columns(df, ratio_threshold=0.8):
    converted = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
        s = df[col]
        if not (pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s)):
            continue
        cleaned = s.astype(str).str.replace(",", "", regex=False).str.strip()
        numeric = pd.to_numeric(cleaned, errors="coerce")
        if numeric.notna().mean() >= ratio_threshold:
            df[col] = numeric
            converted.append(col)
    return converted

df_standardized = df_raw.copy()
df_standardized.columns = make_unique_columns(df_standardized.columns)
df_standardized.columns = [normalize_name(c) for c in df_standardized.columns]
df_standardized.columns = make_unique_columns(df_standardized.columns)

all_empty_cols = [c for c in df_standardized.columns if df_standardized[c].isna().all()]
if all_empty_cols:
    df_standardized = df_standardized.drop(columns=all_empty_cols)

candidate_time_cols = infer_time_columns(df_standardized)
parsed_time_cols = []
for c in candidate_time_cols:
    parsed = pd.to_datetime(df_standardized[c], errors="coerce")
    if parsed.notna().mean() >= 0.5:
        df_standardized[c] = parsed
        parsed_time_cols.append(c)

converted_numeric_cols = coerce_numeric_columns(df_standardized, ratio_threshold=0.8)

rename_map = {}
for c in df_standardized.columns:
    rename_map[c] = re.sub(r"^l(\d+)_", lambda m: f"L{m.group(1)}_", c, flags=re.IGNORECASE)
df_standardized = df_standardized.rename(columns=rename_map)
df_standardized.columns = make_unique_columns(df_standardized.columns)

df_standardized.to_excel(PROCESSED_DATA_PATH, index=False)

print(f"标准化数据已保存：{PROCESSED_DATA_PATH}")
print(f"标准化数据维度：{df_standardized.shape[0]} 行 × {df_standardized.shape[1]} 列")
print(f"删除全空列数量：{len(all_empty_cols)}")
print(f"识别并解析时间列数量：{len(parsed_time_cols)}")
print(f"转换为数值列数量：{len(converted_numeric_cols)}")
display(df_standardized.head())

当前阶段：3. 数据标准化处理层
输入数据：/Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data/equipment_log.xlsx
输出数据：/Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data/processed_equipment_log_standardized.xlsx
标准化数据已保存：/Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data/processed_equipment_log_standardized.xlsx
标准化数据维度：1496 行 × 167 列
删除全空列数量：6
识别并解析时间列数量：8
转换为数值列数量：0


,Time,L1_Mode,L1_Product_Name,L1_Standard_Speed,L1_Target_Speed,L1_Running_Speed,L1_Pump_1_Speed_L,L1_Pump_1_Speed_H,L1_Pump_2_Speed_L,L1_Pump_2_Speed_H,...,RDI_1_Cond,Cu_Strike_Level,Ag_Strike_Level,Ag_Spot_Level,Ag_Spot_pH,Ag_Deplating_1_2_pH,Ag_Deplating_3_pH,RO_1_Pressure,DI_1_Pressure,DI_1_Pressure_1
0,00:00:52,L1:Motion Only,1,0,2.2,0.0,30,85,0,0,...,1.002515,0,0,0,7.066441,8.038797,4.920165,6.012122,1.141699,1.117419
1,00:01:53,L1:Motion Only,1,0,2.2,0.0,30,85,0,0,...,0.997828,0,0,0,7.066441,8.039463,4.920824,5.955643,1.089403,1.043748
2,00:02:54,L1:Motion Only,1,0,2.2,0.0,30,85,0,0,...,0.997828,0,0,0,7.066441,8.038797,4.920824,6.009680,1.140661,1.078405
3,00:03:55,L1:Motion Only,1,0,2.2,0.1,30,85,0,0,...,1.013453,0,0,0,7.066441,8.039463,4.920165,6.019934,1.158923,1.149585
4,00:04:56,L1:Motion Only,1,0,2.2,0.2,30,85,0,0,...,1.013453,0,0,0,7.066441,8.029465,4.920824,6.022701,1.140661,1.162036


In [29]:
print("当前阶段：4. 标准化数据读取与质量检查")
print(f"输入数据：{PROCESSED_DATA_PATH}")

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"未找到标准化后的数据文件：{PROCESSED_DATA_PATH}\n"
        "请先运行数据标准化处理层，生成 processed_equipment_log_standardized.xlsx。"
    )

df_standardized = pd.read_excel(PROCESSED_DATA_PATH)

print(f"使用标准化后的数据文件：{PROCESSED_DATA_PATH}")
print(f"标准化数据维度：{df_standardized.shape[0]} 行 × {df_standardized.shape[1]} 列")
print(f"字段数量：{df_standardized.shape[1]}")

missing_summary = df_standardized.isna().mean().sort_values(ascending=False).head(20)

numeric_cols_all = df_standardized.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols_all = [c for c in df_standardized.columns if c not in numeric_cols_all]

dup_check = pd.Series(df_standardized.columns).value_counts()
dup_check = dup_check[dup_check > 1]

time_cols_qc = []
for c in df_standardized.columns:
    if "time" in str(c).lower() or "date" in str(c).lower():
        parsed = pd.to_datetime(df_standardized[c], errors="coerce")
        if parsed.notna().mean() > 0.5:
            time_cols_qc.append(c)

print(f"数值列数量：{len(numeric_cols_all)}")
print(f"非数值列数量：{len(non_numeric_cols_all)}")
print(f"重复列数量：{dup_check.shape[0]}")
print(f"可用时间列数量：{len(time_cols_qc)}")

display(df_standardized.head())
display(missing_summary.to_frame("missing_ratio"))

当前阶段：4. 标准化数据读取与质量检查
输入数据：/Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data/processed_equipment_log_standardized.xlsx
使用标准化后的数据文件：/Users/jiaoguodong/Desktop/myApp/Industry_AIDemo/data/processed_equipment_log_standardized.xlsx
标准化数据维度：1496 行 × 167 列
字段数量：167
数值列数量：134
非数值列数量：33
重复列数量：0
可用时间列数量：9


,Time,L1_Mode,L1_Product_Name,L1_Standard_Speed,L1_Target_Speed,L1_Running_Speed,L1_Pump_1_Speed_L,L1_Pump_1_Speed_H,L1_Pump_2_Speed_L,L1_Pump_2_Speed_H,...,RDI_1_Cond,Cu_Strike_Level,Ag_Strike_Level,Ag_Spot_Level,Ag_Spot_pH,Ag_Deplating_1_2_pH,Ag_Deplating_3_pH,RO_1_Pressure,DI_1_Pressure,DI_1_Pressure_1
0,00:00:52,L1:Motion Only,1,0,2.2,0.0,30,85,0,0,...,1.002515,0,0,0,7.066441,8.038797,4.920165,6.012122,1.141699,1.117419
1,00:01:53,L1:Motion Only,1,0,2.2,0.0,30,85,0,0,...,0.997828,0,0,0,7.066441,8.039463,4.920824,5.955643,1.089403,1.043748
2,00:02:54,L1:Motion Only,1,0,2.2,0.0,30,85,0,0,...,0.997828,0,0,0,7.066441,8.038797,4.920824,6.009680,1.140661,1.078405
3,00:03:55,L1:Motion Only,1,0,2.2,0.1,30,85,0,0,...,1.013453,0,0,0,7.066441,8.039463,4.920165,6.019934,1.158923,1.149585
4,00:04:56,L1:Motion Only,1,0,2.2,0.2,30,85,0,0,...,1.013453,0,0,0,7.066441,8.029465,4.920824,6.022701,1.140661,1.162036


,missing_ratio
Time,0.0
L1_Mode,0.0
L1_Product_Name,0.0
L1_Standard_Speed,0.0
L1_Target_Speed,0.0
L1_Running_Speed,0.0
L1_Pump_1_Speed_L,0.0
L1_Pump_1_Speed_H,0.0
L1_Pump_2_Speed_L,0.0
L1_Pump_2_Speed_H,0.0


## 4) 生产运行状态概览：设备共有状态、L1、L2 同步观察

在正式进入特征选择、异常检测和 AI 洞察之前，本节先基于标准化后的设备数据，对生产运行状态进行客观观察。

本数据由三类同步时间序列组成：

1. **设备共有状态参数**：反映整台设备在当前时间点的共有运行状态；
2. **L1 产线状态参数**：反映 L1 产线在当前时间点的工艺状态；
3. **L2 产线状态参数**：反映 L2 产线在当前时间点的工艺状态。

三类数据共享同一时间轴，因此可以在同一时间点下进行对齐观察。

本节重点通过趋势图、双产线对比图、差值图、联动趋势图和热力图观察：

- 设备共有状态是否平稳；
- L1 与 L2 各自是否平稳；
- L1 与 L2 是否同步；
- L1 与 L2 是否在某些时间段出现分化；
- 设备共有状态变化时，双产线状态是否也出现同步变化；
- 哪些时间段值得后续异常检测重点关注。

本节只做客观观察，不直接下因果结论。


In [30]:
from sklearn.preprocessing import StandardScaler

print("当前阶段：4A. 生产运行状态概览")
print(f"输入数据：{PROCESSED_DATA_PATH}")

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"未找到标准化后的数据文件：{PROCESSED_DATA_PATH}\n"
        "请先运行前面的数据标准化处理步骤。"
    )

df_state = pd.read_excel(PROCESSED_DATA_PATH)

print(f"标准化数据维度：{df_state.shape[0]} 行 × {df_state.shape[1]} 列")
display(df_state.head())

# 时间列识别
_time_keywords = ["time", "timestamp", "datetime", "date", "时间", "日期"]
score_cols = []
for c in df_state.columns:
    cl = str(c).lower()
    score = sum(1 for k in _time_keywords if k in cl)
    if score > 0:
        parsed = pd.to_datetime(df_state[c], errors="coerce")
        score_cols.append((c, score, parsed.notna().mean()))

if not score_cols:
    raise ValueError("未能识别时间列，请检查标准化数据中的时间字段命名。")

time_col = sorted(score_cols, key=lambda x: (x[1], x[2]), reverse=True)[0][0]

df_state[time_col] = pd.to_datetime(df_state[time_col], errors="coerce")
df_state = df_state.sort_values(time_col).reset_index(drop=True)

# 三类参数列识别
_l1_patterns = ["l1", "line1", "line 1", "产线1"]
_l2_patterns = ["l2", "line2", "line 2", "产线2"]

l1_cols = [c for c in df_state.columns if c != time_col and any(p in str(c).lower() for p in _l1_patterns)]
l2_cols = [c for c in df_state.columns if c != time_col and any(p in str(c).lower() for p in _l2_patterns)]
common_state_cols = [c for c in df_state.columns if c not in set([time_col, *l1_cols, *l2_cols])]

print(f"时间列：{time_col}")
print(f"设备共有状态参数列数量：{len(common_state_cols)}")
print(f"L1 产线参数列数量：{len(l1_cols)}")
print(f"L2 产线参数列数量：{len(l2_cols)}")

# L1/L2 同名参数映射
prefix_re = re.compile(r"^(l1|l2|line\s*1|line\s*2|产线1|产线2)[_\-\s]*", re.IGNORECASE)

def base_name(col):
    b = prefix_re.sub("", str(col)).strip()
    return re.sub(r"\s+", " ", b)

l1_map = {base_name(c): c for c in l1_cols}
l2_map = {base_name(c): c for c in l2_cols}

paired_features = [
    {"base_name": b, "l1_col": l1_map[b], "l2_col": l2_map[b]}
    for b in sorted(set(l1_map) & set(l2_map))
]
l1_only_features = sorted(list(set(l1_map) - set(l2_map)))
l2_only_features = sorted(list(set(l2_map) - set(l1_map)))

print(f"L1/L2 可配对参数数量：{len(paired_features)}")
print(f"L1 专属字段数量：{len(l1_only_features)}")
print(f"L2 专属字段数量：{len(l2_only_features)}")
print("L2 专属字段：", l2_only_features)

# 选择工具函数

def top_numeric_by_variation(df, cols, top_n=5, max_missing=0.4):
    numeric = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric:
        return []
    stats = []
    for c in numeric:
        miss = df[c].isna().mean()
        uniq = df[c].nunique(dropna=True)
        std = df[c].std(skipna=True)
        if miss <= max_missing and uniq > 1 and pd.notna(std):
            stats.append((c, std, miss))
    stats = sorted(stats, key=lambda x: (x[1], -x[2]), reverse=True)
    return [x[0] for x in stats[:top_n]]

selected_common_state_cols = top_numeric_by_variation(df_state, common_state_cols, top_n=5)

# 图1
if selected_common_state_cols:
    fig, axes = plt.subplots(len(selected_common_state_cols), 1, figsize=(14, 3 * len(selected_common_state_cols)), sharex=True)
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, selected_common_state_cols):
        ax.plot(df_state[time_col], df_state[col], lw=1.2)
        ax.set_title(f"设备共有状态趋势：{col}")
        ax.set_ylabel("参数值")
        ax.grid(alpha=0.2)
    axes[-1].set_xlabel("时间")
    plt.tight_layout(); plt.show()
print("该图用于观察设备共有状态参数在同一时间轴上的变化情况，帮助判断设备整体背景状态是否平稳。")

# 选择配对参数
keywords = ["tc", "temp", "temperature", "current", "voltage", "flow", "ph", "speed", "pressure", "onoff"]
pair_stats = []
for p in paired_features:
    l1c, l2c = p['l1_col'], p['l2_col']
    if pd.api.types.is_numeric_dtype(df_state[l1c]) and pd.api.types.is_numeric_dtype(df_state[l2c]):
        m = max(df_state[l1c].isna().mean(), df_state[l2c].isna().mean())
        if m <= 0.4 and df_state[l1c].nunique(dropna=True) > 1 and df_state[l2c].nunique(dropna=True) > 1:
            var_score = float(df_state[l1c].var(skipna=True) + df_state[l2c].var(skipna=True))
            kw = any(k in p['base_name'].lower() for k in keywords)
            pair_stats.append((p, kw, var_score))
pair_stats = sorted(pair_stats, key=lambda x: (x[1], x[2]), reverse=True)
selected_pairs_for_trend = [x[0] for x in pair_stats[:6]]
if len(selected_pairs_for_trend) < 4:
    selected_pairs_for_trend = [x[0] for x in sorted(pair_stats, key=lambda x: x[2], reverse=True)[:6]]

# 图2
if selected_pairs_for_trend:
    fig, axes = plt.subplots(len(selected_pairs_for_trend), 1, figsize=(14, 3.2 * len(selected_pairs_for_trend)), sharex=True)
    axes = np.array(axes).reshape(-1)
    for ax, p in zip(axes, selected_pairs_for_trend):
        ax.plot(df_state[time_col], df_state[p['l1_col']], label='L1', lw=1.1)
        ax.plot(df_state[time_col], df_state[p['l2_col']], label='L2', lw=1.1)
        ax.set_title(f"双产线同名参数趋势：{p['base_name']}")
        ax.set_ylabel('参数值'); ax.legend(); ax.grid(alpha=0.2)
    axes[-1].set_xlabel('时间')
    plt.tight_layout(); plt.show()
print("该图用于观察 L1 与 L2 在同一时间点下的同名参数走势是否一致，以及是否存在局部分化。")

# 图3
if selected_pairs_for_trend:
    fig, axes = plt.subplots(len(selected_pairs_for_trend), 1, figsize=(14, 3.2 * len(selected_pairs_for_trend)), sharex=True)
    axes = np.array(axes).reshape(-1)
    for ax, p in zip(axes, selected_pairs_for_trend):
        diff = df_state[p['l2_col']] - df_state[p['l1_col']]
        ax.plot(df_state[time_col], diff, color='tab:purple', lw=1.0)
        ax.axhline(0, color='black', ls='--', lw=0.8)
        ax.set_title(f"双产线差值趋势：L2 - L1 ({p['base_name']})")
        ax.set_ylabel('L2-L1'); ax.grid(alpha=0.2)
    axes[-1].set_xlabel('时间')
    plt.tight_layout(); plt.show()
print("该图用于观察同一时间点下 L2 相对 L1 的偏移情况。差值长期偏离 0 或在局部时间段突然扩大，说明双产线状态可能出现分化。")

# 图4
if selected_common_state_cols and selected_pairs_for_trend:
    ccol = selected_common_state_cols[0]
    p = selected_pairs_for_trend[0]
    fig, axes = plt.subplots(2,1,figsize=(14,7),sharex=True)
    axes[0].plot(df_state[time_col], df_state[ccol], color='tab:green')
    axes[0].set_title(f"设备共有状态与双产线参数联动观察（设备参数：{ccol}）")
    axes[0].set_ylabel(ccol); axes[0].grid(alpha=0.2)
    axes[1].plot(df_state[time_col], df_state[p['l1_col']], label='L1', lw=1.1)
    axes[1].plot(df_state[time_col], df_state[p['l2_col']], label='L2', lw=1.1)
    axes[1].set_ylabel(p['base_name']); axes[1].legend(); axes[1].grid(alpha=0.2)
    axes[1].set_xlabel('时间')
    plt.tight_layout(); plt.show()
print("该图用于观察设备共有状态变化与 L1/L2 产线状态变化在时间上的同步关系。该观察仅表示同步变化，不代表因果关系。")

# 图5 热力图

def pick_heatmap_cols(df, cols, n=8):
    return top_numeric_by_variation(df, cols, top_n=n, max_missing=0.5)

selected_l1_heatmap_cols = pick_heatmap_cols(df_state, l1_cols, n=10)
selected_l2_heatmap_cols = pick_heatmap_cols(df_state, l2_cols, n=10)
selected_common_heatmap_cols = pick_heatmap_cols(df_state, common_state_cols, n=8)

fig, axes = plt.subplots(3,1,figsize=(16,10),sharex=True)
for ax, cols, ttl in [
    (axes[0], selected_common_heatmap_cols, '设备共有状态热力图'),
    (axes[1], selected_l1_heatmap_cols, 'L1 产线状态热力图'),
    (axes[2], selected_l2_heatmap_cols, 'L2 产线状态热力图')
]:
    if cols:
        mat = df_state[cols].copy().apply(pd.to_numeric, errors='coerce')
        mat = mat.fillna(mat.median(numeric_only=True))
        z = StandardScaler().fit_transform(mat)
        im = ax.imshow(z.T, aspect='auto', cmap='coolwarm', interpolation='nearest')
        ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
    ax.set_title(ttl)
axes[-1].set_xlabel('时间序列索引')
plt.tight_layout(); plt.show()
print("热力图用于观察多参数是否在同一时间段出现整体偏高、偏低或状态切换。若三层热力图在相近时间段同时出现颜色变化，说明设备共有状态和双产线状态可能存在同步状态变化，建议在后续异常检测中重点关注。")

# 图6 滚动波动
rolling_pairs = selected_pairs_for_trend[:4]
if rolling_pairs:
    fig, axes = plt.subplots(len(rolling_pairs),1,figsize=(14,3*len(rolling_pairs)),sharex=True)
    axes = np.array(axes).reshape(-1)
    for ax, p in zip(axes, rolling_pairs):
        r1 = df_state[p['l1_col']].rolling(window=20,min_periods=5).std()
        r2 = df_state[p['l2_col']].rolling(window=20,min_periods=5).std()
        ax.plot(df_state[time_col], r1, label='L1 rolling std', lw=1.0)
        ax.plot(df_state[time_col], r2, label='L2 rolling std', lw=1.0)
        ax.set_title(f"双产线滚动波动对比：{p['base_name']}")
        ax.set_ylabel('rolling std'); ax.legend(); ax.grid(alpha=0.2)
    axes[-1].set_xlabel('时间')
    plt.tight_layout(); plt.show()
print("滚动波动图用于观察过程稳定性。如果某一时间段滚动标准差上升，说明该阶段参数波动加大，值得在后续分析中关注。")

# 客观总结
common_var_top = df_state[selected_common_state_cols].std(skipna=True).sort_values(ascending=False).head(3).index.tolist() if selected_common_state_cols else []
diff_stats = []
for p in selected_pairs_for_trend:
    d = df_state[p['l2_col']] - df_state[p['l1_col']]
    diff_stats.append((p['base_name'], d.abs().mean(skipna=True), d.std(skipna=True)))
diff_mean_top = [x[0] for x in sorted(diff_stats, key=lambda x: x[1], reverse=True)[:3]]
diff_std_top = [x[0] for x in sorted(diff_stats, key=lambda x: x[2], reverse=True)[:3]]

production_state_observations = [
    "### 生产状态客观观察总结",
    "- 本节基于标准化后的同一时间轴数据，对设备共有状态、L1 产线状态和 L2 产线状态进行了同步观察。",
    f"- 设备共有状态参数中，{('、'.join(common_var_top) if common_var_top else '暂无可用字段')} 的波动相对更明显，建议在后续异常检测中作为设备背景状态重点关注。",
    f"- 从 L2 - L1 差值趋势看，{('、'.join(diff_mean_top) if diff_mean_top else '暂无可用字段')} 的平均偏移较明显，可作为观察双产线分化的候选指标。",
    f"- 从差值波动看，{('、'.join(diff_std_top) if diff_std_top else '暂无可用字段')} 在局部时间段更可能出现差值变化扩大。",
    "- 热力图显示，部分时间段多组参数出现同步偏高或偏低现象，建议在后续异常检测中进一步验证这些时间窗口。",
    "- 以上观察仅表示生产状态变化和同步现象，不代表因果结论。"
]
print("
".join(production_state_observations))


SyntaxError: unterminated string literal (detected at line 219) (3881949006.py, line 219)

In [ ]:
print("当前阶段：5. 特征选择与清洗层")
print(f"输入数据：{PROCESSED_DATA_PATH}")

df_clean = df_standardized.copy()
df_clean = df_clean.dropna(axis=1, how="all")

possible_time_cols = [c for c in df_clean.columns if any(k in str(c).lower() for k in ["time", "date", "timestamp"])]
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

constant_cols = [col for col in numeric_cols if df_clean[col].nunique(dropna=True) <= 1]
if constant_cols:
    df_clean = df_clean.drop(columns=constant_cols)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
missing_ratio = df_clean[numeric_cols].isna().mean() if numeric_cols else pd.Series(dtype=float)
feature_cols = missing_ratio[missing_ratio < 0.5].index.tolist() if not missing_ratio.empty else []

if not feature_cols:
    raise ValueError("清洗后没有可用数值特征，请检查原始数据或放宽筛选阈值。")

X_base = df_clean[feature_cols].copy()
X_base = X_base.replace([np.inf, -np.inf], np.nan)
X_base = X_base.fillna(X_base.median(numeric_only=True))

print(f"清洗后数据维度：{df_clean.shape}")
print(f"候选数值特征数量：{len(feature_cols)}")
print(f"删除常数列数量：{len(constant_cols)}")
print(f"识别的时间列候选数量：{len(possible_time_cols)}")
print("前 20 个候选特征：", feature_cols[:20])

In [ ]:
print("当前阶段：6. 特征工程层")
print(f"输入特征数量：{len(feature_cols)}")

from sklearn.preprocessing import StandardScaler

feature_df = df_clean.copy()
X_base = X_base.replace([np.inf, -np.inf], np.nan)
X_base = X_base.fillna(X_base.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_base)
X = pd.DataFrame(X_scaled, columns=feature_cols, index=X_base.index)

TOP_N_FOR_ROLLING = min(20, len(feature_cols))
rolling_base_cols = feature_cols[:TOP_N_FOR_ROLLING]
for col in rolling_base_cols:
    X[f"{col}__diff1"] = X[col].diff().fillna(0.0)
    X[f"{col}__roll_mean_5"] = X[col].rolling(window=5, min_periods=1).mean()
    X[f"{col}__roll_std_5"] = X[col].rolling(window=5, min_periods=1).std().fillna(0.0)

engineered_feature_cols = X.columns.tolist()

print(f"特征工程后特征矩阵维度：{X.shape}")
print(f"工程化特征总数：{len(engineered_feature_cols)}")

In [ ]:
print("当前阶段：7. 异常检测层")
print("输入数据：特征工程输出矩阵 X")

from sklearn.ensemble import IsolationForest

model = IsolationForest(n_estimators=200, contamination=0.1, random_state=42)
anomaly_pred = model.fit_predict(X)
anomaly_score = model.decision_function(X)

df_anomaly = feature_df.copy()
df_anomaly["anomaly_label"] = np.where(anomaly_pred == -1, 1, 0)
df_anomaly["anomaly_score"] = anomaly_score

print("异常检测完成")
print(df_anomaly["anomaly_label"].value_counts())
print(f"异常比例：{df_anomaly['anomaly_label'].mean():.2%}")

In [ ]:
print("当前阶段：8. 分段对比与核心洞察层")
print("输入数据：df_anomaly + engineered_feature_cols")

normal_df = df_anomaly[df_anomaly["anomaly_label"] == 0]
abnormal_df = df_anomaly[df_anomaly["anomaly_label"] == 1]

comparison_rows = []
base_compare_cols = [c for c in feature_cols if c in df_anomaly.columns]

for col in base_compare_cols:
    normal_mean = normal_df[col].mean()
    abnormal_mean = abnormal_df[col].mean()
    normal_std = normal_df[col].std()
    abnormal_std = abnormal_df[col].std()
    diff = abnormal_mean - normal_mean
    pct_change = diff / (abs(normal_mean) + 1e-9)

    comparison_rows.append({
        "feature": col,
        "normal_mean": normal_mean,
        "abnormal_mean": abnormal_mean,
        "mean_diff": diff,
        "pct_change": pct_change,
        "normal_std": normal_std,
        "abnormal_std": abnormal_std,
        "std_diff": abnormal_std - normal_std,
        "abs_pct_change": abs(pct_change)
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("abs_pct_change", ascending=False)
top_factors = comparison_df.head(10).copy()
insight_table = top_factors.copy()

print(f"参与对比特征数量：{len(base_compare_cols)}")
print("Top 10 异常关联因子：")
display(top_factors)

In [ ]:
print("当前阶段：9. 可视化展示层")
print("输入数据：df_anomaly / comparison_df / top_factors")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(df_anomaly["anomaly_score"], bins=30, color="#4C78A8", alpha=0.85)
axes[0, 0].set_title("异常分数分布")
axes[0, 0].set_xlabel("anomaly_score")
axes[0, 0].set_ylabel("count")

label_counts = df_anomaly["anomaly_label"].value_counts().sort_index()
axes[0, 1].bar(["正常(0)", "异常(1)"], [label_counts.get(0, 0), label_counts.get(1, 0)], color=["#72B7B2", "#E45756"])
axes[0, 1].set_title("正常/异常样本数量")

plot_df = top_factors.sort_values("abs_pct_change", ascending=True)
axes[1, 0].barh(plot_df["feature"], plot_df["abs_pct_change"], color="#F58518")
axes[1, 0].set_title("Top 10 关键差异因子（绝对变化比例）")
axes[1, 0].set_xlabel("abs_pct_change")

time_col = None
for c in df_anomaly.columns:
    if any(k in str(c).lower() for k in ["time", "date", "timestamp"]):
        parsed = pd.to_datetime(df_anomaly[c], errors="coerce")
        if parsed.notna().mean() > 0.5:
            time_col = c
            break

if time_col is not None:
    temp = df_anomaly.copy()
    temp[time_col] = pd.to_datetime(temp[time_col], errors="coerce")
    temp = temp.dropna(subset=[time_col]).sort_values(time_col)
    axes[1, 1].plot(temp[time_col], temp["anomaly_label"], marker=".", linestyle="", alpha=0.5)
    axes[1, 1].set_title(f"异常点时间分布（{time_col}）")
    axes[1, 1].set_xlabel("time")
    axes[1, 1].set_ylabel("anomaly_label")
else:
    axes[1, 1].text(0.1, 0.5, "未识别到可用时间列", fontsize=12)
    axes[1, 1].set_title("异常随时间分布")
    axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("当前阶段：10. AI 文本洞察生成层")
print("输入数据：analysis_summary（结构化结果）")

analysis_summary = {
    "data_shape": df_standardized.shape,
    "clean_shape": df_clean.shape,
    "feature_count": len(feature_cols),
    "anomaly_count": int(df_anomaly["anomaly_label"].sum()),
    "anomaly_ratio": float(df_anomaly["anomaly_label"].mean()),
    "top_factors": top_factors.to_dict(orient="records"),
    "missing_top10": df_standardized.isna().mean().sort_values(ascending=False).head(10).round(4).to_dict()
}

prompt = (
    "请基于以下结构化分析结果生成‘AI 良率异常洞察报告’。\n"
    "要求：\n"
    "1) 不要编造未计算出的数据；\n"
    "2) 不要直接给出因果结论；\n"
    "3) 使用‘可能相关/优先排查/建议进一步验证’等措辞；\n"
    "4) 报告受众为工程师和管理者；\n"
    "5) 使用如下结构：异常概况、关键异常关联因子、工艺解释、优先排查建议、风险提示、下一步数据补充建议。\n\n"
    f"结构化输入：\n{analysis_summary}"
)

ai_report = None
try:
    import sys
    sys.path.append(str(Path("src").resolve()))
    from llm_provider import generate_insight

    llm_result = generate_insight(prompt=prompt, mode="auto")
    ai_report = llm_result.get("report_md") or llm_result.get("text")
    if ai_report:
        print("LLM 生成成功（若不可用将自动回退）。")
        print(ai_report)
except Exception as e:
    print(f"LLM 调用不可用，进入规则生成 fallback：{e}")

if not ai_report:
    fallback_report = f"""
## AI 良率异常洞察报告（规则生成版）

本次分析基于标准化后的设备 Log 数据，共包含 {df_standardized.shape[0]} 行、{df_standardized.shape[1]} 列。
经过特征清洗后，保留 {len(feature_cols)} 个候选工艺特征。

异常检测识别出 {int(df_anomaly["anomaly_label"].sum())} 条疑似异常记录，
异常比例为 {df_anomaly["anomaly_label"].mean():.2%}。

从正常段与异常段对比看，以下参数差异最明显：

{top_factors[["feature", "normal_mean", "abnormal_mean", "pct_change"]].to_markdown(index=False)}

这些参数可作为后续工程排查的优先方向。
需要注意的是，本分析结果表示异常相关性，不等同于严格因果结论。
"""
    ai_report = fallback_report
    print(ai_report)

## 11) 总结与下一步扩展

本 Notebook 已完成如下闭环：

```text
原始设备 Log
    → 标准化数据表
    → 特征清洗
    → 特征工程
    → 异常检测
    → 分段对比
    → AI 洞察报告
```

下一步可扩展方向：

- 接入真实良率标签；
- 接入批次号 / 产品型号 / 设备号；
- 接入班次 / 操作员 / 药液批次；
- 从异常检测升级为良率预测；
- 从单次分析升级为每日自动报告；
- 与 MES / ERP / 数据平台集成；
- 沉淀为可复用的 AI 良率分析模块。